In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/pickles1/difficult_range.pkl
/kaggle/input/pickles1/difficult_samples.pkl
/kaggle/input/pickles1/difficult_max.pkl
/kaggle/input/pickles1/difficult_min.pkl
/kaggle/input/unsw-nb15/UNSW_NB15_testing-set.csv
/kaggle/input/unsw-nb15/UNSW-NB15_1.csv
/kaggle/input/unsw-nb15/UNSW_NB15_training-set.csv
/kaggle/input/unsw-nb15/UNSW-NB15_LIST_EVENTS.csv
/kaggle/input/unsw-nb15/UNSW-NB15_4.csv
/kaggle/input/unsw-nb15/UNSW-NB15_3.csv
/kaggle/input/unsw-nb15/UNSW-NB15_2.csv
/kaggle/input/unsw-nb15/NUSW-NB15_features.csv


In [2]:
import pandas as pd
import numpy as np
import sys
import keras
import sklearn
from keras.models import Sequential
from keras.layers import Dense, Dropout, Activation, Embedding, Flatten
from keras.layers import LSTM, SimpleRNN, GRU, Bidirectional, BatchNormalization,Convolution1D,MaxPooling1D, Reshape, GlobalAveragePooling1D
from keras.utils import to_categorical
import sklearn.preprocessing
from sklearn import metrics
from scipy.stats import zscore
from tensorflow.keras.utils import get_file, plot_model
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt
from imblearn.over_sampling import SMOTE

print(pd.__version__)
print(np.__version__)
print(sys.version)
print(sklearn.__version__)

2.0.3
1.24.3
3.10.12 | packaged by conda-forge | (main, Jun 23 2023, 22:40:32) [GCC 12.3.0]
1.2.2


In [3]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score

In [4]:
import pickle

In [5]:
# open("Fruits.obj",'rb')

In [6]:
# diff_samples = open('../input/pickles1/difficult_samples.pkl','rb')
# diff_samples

In [7]:
# diff_max = open('../input/pickles1/difficult_max.pkl','rb')
# diff_max

In [8]:
# diff_min = open('../input/pickles1/difficult_min.pkl','rb')
# diff_min

In [9]:
# diff_range = open('../input/pickles1/difficult_range.pkl','rb')
# diff_range

In [10]:
import pandas as pd

# Load data from pickle files
diff_samples = pd.read_pickle('../input/pickles1/difficult_samples.pkl')
diff_min = pd.read_pickle('../input/pickles1/difficult_min.pkl')  # Replace <path_to_diff_samples_pickle_file> with the correct path

In [11]:

# Load data from pickle files
diff_range = pd.read_pickle('../input/pickles1/difficult_range.pkl')
diff_max = pd.read_pickle('../input/pickles1/difficult_max.pkl')  # Replace <path_to_diff_samples_pickle_file> with the correct path

In [12]:
# percentage_to_remove = 0.1  # for example, 10%

# # Calculate the number of rows to remove
# num_rows_to_remove = int(len(diff_range) * percentage_to_remove)

# # Randomly select rows to remove
# rows_to_remove = diff_range.sample(n=num_rows_to_remove, random_state=42)  # Set random_state for reproducibility

# # Remove selected rows from the DataFrame
# diff_range_cleaned = diff_range.drop(rows_to_remove.index)

# # Reset the index of the cleaned DataFrame
# diff_range_cleaned.reset_index(drop=True, inplace=True)

# # Check for duplicates in the cleaned DataFrame
# print("Duplicates in diff_range_cleaned:", diff_range_cleaned.index.duplicated().any())

In [13]:
diff_samples

,dur,spkts,dpkts,sbytes,dbytes,rate,sttl,dttl,sload,dload,...,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,proto,service,state,proto,service,state,attack_cat
0,0.576920,0.088882,0.472555,0.327664,0.628358,0.890093,0.120413,0.202092,4.852415e-01,0.931179,...,0.933748,0.615069,0.787921,unas,-,INT,0.7104814773047498,0.4935289788472922,0.5469086150223595,Backdoor
1,0.000003,2.000000,0.000000,200.000000,0.000000,333333.321500,254.000000,0.000000,2.666667e+08,0.000000,...,2.000000,6.000000,0.000000,unas,-,INT,NaN,NaN,NaN,Backdoor
2,0.000008,2.000000,0.000000,200.000000,0.000000,125000.000300,254.000000,0.000000,1.000000e+08,0.000000,...,11.000000,4.000000,0.000000,unas,-,INT,NaN,NaN,NaN,Backdoor
3,0.000008,2.000000,0.000000,200.000000,0.000000,125000.000300,254.000000,0.000000,1.000000e+08,0.000000,...,11.000000,4.000000,0.000000,sat-expak,-,INT,NaN,NaN,NaN,Backdoor
4,0.000008,2.000000,0.000000,200.000000,0.000000,125000.000300,254.000000,0.000000,1.000000e+08,0.000000,...,11.000000,4.000000,0.000000,sm,-,INT,NaN,NaN,NaN,Backdoor
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1171515,59.400505,64.000000,0.000000,17408.000000,0.000000,1.060597,254.000000,0.000000,2.307859e+03,0.000000,...,3.000000,2.000000,0.000000,emcon,-,INT,NaN,NaN,NaN,Shellcode
1171516,0.294185,10.000000,6.000000,582.000000,268.000000,50.988321,254.000000,252.000000,1.424954e+04,6091.404785,...,1.000000,1.000000,0.000000,unas,-,INT,NaN,NaN,NaN,Shellcode
1171517,1.227135,4.000000,2.000000,568.000000,432.000000,4.074532,254.000000,60.000000,2.777201e+03,1408.158081,...,2.000000,2.000000,0.000000,unas,-,INT,NaN,NaN,NaN,Shellcode
1171518,1.227135,4.000000,2.000000,568.000000,432.000000,4.074532,254.000000,60.000000,2.777201e+03,1408.158081,...,2.000000,2.000000,0.000000,unas,-,INT,NaN,NaN,NaN,Shellcode


In [14]:
# diff_range

In [15]:
# diff_range.drop(['dur ','spkts ','dpkts','sbytes','dbytes','rate' ,'sttl ','dttl ','sload ','dload ','sloss','dloss','sinpkt','dinpkt','sjit','djit','swin','stcpb','dtcpb','dwin','tcprtt','synack','ackdat','smean','dmean','trans_depth','response_body_len','ct_srv_src','ct_state_ttl','ct_dst_ltm ','ct_src_dport_ltm','ct_dst_sport_ltm ','ct_dst_src_ltm' ,'is_ftp_login ','ct_ftp_cmd ','ct_flw_http_mthd','ct_src_ltm ','ct_srv_dst ','is_sm_ips_ports'], axis=1)

In [16]:
# for col in diff_range.columns:
#     print(col)

In [17]:
columns_to_drop = ['proto','service','state']

# Drop specified columns from the DataFrame
diff_samples_cleaned = diff_samples.drop(columns=columns_to_drop)

In [18]:
diff_samples_cleaned

,dur,spkts,dpkts,sbytes,dbytes,rate,sttl,dttl,sload,dload,...,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,attack_cat
0,0.576920,0.088882,0.472555,0.327664,0.628358,0.890093,0.120413,0.202092,4.852415e-01,0.931179,...,0.15565,0.588003,0.521122,0.064124,0.031731,0.096953,0.933748,0.615069,0.787921,Backdoor
1,0.000003,2.000000,0.000000,200.000000,0.000000,333333.321500,254.000000,0.000000,2.666667e+08,0.000000,...,2.00000,2.000000,7.000000,0.000000,0.000000,0.000000,2.000000,6.000000,0.000000,Backdoor
2,0.000008,2.000000,0.000000,200.000000,0.000000,125000.000300,254.000000,0.000000,1.000000e+08,0.000000,...,1.00000,1.000000,4.000000,0.000000,0.000000,0.000000,11.000000,4.000000,0.000000,Backdoor
3,0.000008,2.000000,0.000000,200.000000,0.000000,125000.000300,254.000000,0.000000,1.000000e+08,0.000000,...,1.00000,1.000000,4.000000,0.000000,0.000000,0.000000,11.000000,4.000000,0.000000,Backdoor
4,0.000008,2.000000,0.000000,200.000000,0.000000,125000.000300,254.000000,0.000000,1.000000e+08,0.000000,...,1.00000,1.000000,4.000000,0.000000,0.000000,0.000000,11.000000,4.000000,0.000000,Backdoor
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1171515,59.400505,64.000000,0.000000,17408.000000,0.000000,1.060597,254.000000,0.000000,2.307859e+03,0.000000,...,2.00000,2.000000,5.000000,0.000000,0.000000,0.000000,3.000000,2.000000,0.000000,Shellcode
1171516,0.294185,10.000000,6.000000,582.000000,268.000000,50.988321,254.000000,252.000000,1.424954e+04,6091.404785,...,1.00000,1.000000,1.000000,0.000000,0.000000,0.000000,1.000000,1.000000,0.000000,Shellcode
1171517,1.227135,4.000000,2.000000,568.000000,432.000000,4.074532,254.000000,60.000000,2.777201e+03,1408.158081,...,2.00000,2.000000,2.000000,0.000000,0.000000,0.000000,2.000000,2.000000,0.000000,Shellcode
1171518,1.227135,4.000000,2.000000,568.000000,432.000000,4.074532,254.000000,60.000000,2.777201e+03,1408.158081,...,2.00000,2.000000,2.000000,0.000000,0.000000,0.000000,2.000000,2.000000,0.000000,Shellcode


In [19]:

# Concatenate dataframes without ignoring the index
new_train_set = pd.concat([diff_samples_cleaned, diff_range,diff_min,diff_max])

# Reset the index for the concatenated dataframe
# new_train_set.reset_index(drop=True, inplace=True)

# # Check for duplicates in the concatenated dataframe
# print("Duplicates in new_train_set:", new_train_set.index.duplicated().any())

# # Shuffle the concatenated dataframe
df2 = new_train_set.sample(frac=1).reset_index(drop=True)

In [20]:
df2

,dur,spkts,dpkts,sbytes,dbytes,rate,sttl,dttl,sload,dload,...,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,attack_cat,proto,service,state
0,1.728629,10.0,8.0,780.0,354.0,9.834383,254.0,252.0,3.248817e+03,1434.662964,...,0.0,0.0,0.0,2.0,1.0,0.0,Backdoor,NaN,NaN,NaN
1,0.496700,10.0,6.0,534.0,268.0,30.199316,254.0,252.0,7.747131e+03,3607.811523,...,0.0,0.0,0.0,3.0,3.0,0.0,Normal,tcp,-,FIN
2,0.000009,2.0,0.0,120.0,0.0,111111.107200,254.0,0.0,5.333333e+07,0.000000,...,0.0,0.0,0.0,2.0,1.0,0.0,Shellcode,udp,-,INT
3,0.000003,2.0,0.0,200.0,0.0,333333.321500,254.0,0.0,2.666667e+08,0.000000,...,0.0,0.0,0.0,3.0,5.0,0.0,Shellcode,NaN,NaN,NaN
4,0.000009,2.0,0.0,200.0,0.0,111111.107200,254.0,0.0,8.888889e+07,0.000000,...,0.0,0.0,0.0,2.0,5.0,0.0,Shellcode,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2028143,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2028144,0.000006,2.0,0.0,200.0,0.0,166666.660800,254.0,0.0,1.333333e+08,0.000000,...,0.0,0.0,0.0,6.0,7.0,0.0,Shellcode,NaN,NaN,NaN
2028145,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2028146,0.000009,2.0,0.0,200.0,0.0,111111.107200,254.0,0.0,8.888889e+07,0.000000,...,0.0,0.0,0.0,23.0,5.0,0.0,Worms,NaN,NaN,NaN


In [21]:
from sklearn.model_selection import train_test_split

In [22]:

# unique_labels = le.transform(df2['attack_cat'].unique())

In [23]:


X_train, X_test, y_train, y_test = train_test_split(df2.iloc[:,:-1], df2.iloc[:,-1], test_size=0.3, random_state=42)
     

to_remove = [x for x in y_test.unique() if x not in y_train.unique()]
rows_to_remove = y_test.isin(to_remove)

y_test = y_test[~rows_to_remove]
X_test = X_test[~rows_to_remove]
     

X_train.shape

(1419703, 42)

In [24]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Identify categorical columns
categorical_columns = ['attack_cat']

categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', categorical_transformer, categorical_columns)
    ])

# Set with_mean=False for StandardScaler to handle sparse matrices
pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                           ('scaler', StandardScaler(with_mean=False))])

# Apply preprocessing to training and testing sets
X_train_scaled = pipeline.fit_transform(X_train)
X_test_scaled = pipeline.transform(X_test)


In [25]:
target_encoder = OneHotEncoder(handle_unknown='ignore', sparse=False)
y_train_encoded = target_encoder.fit_transform(y_train.values.reshape(-1, 1))
y_test_encoded = target_encoder.transform(y_test.values.reshape(-1, 1))


/opt/conda/lib/python3.10/site-packages/sklearn/preprocessing/_encoders.py:868: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


In [26]:

# le = LabelEncoder()
# y_train_encoded = le.fit_transform(y_train)
# y_test_encoded = le.transform(y_test)

In [27]:
# unique_labels = le.transform(df2['attack_cat'].unique())

In [28]:
model = Sequential()
model.add(Dense(64, activation='relu', input_dim=X_train_scaled.shape[1]))
model.add(Dense(32, activation='relu'))
model.add(Dense(y_train_encoded.shape[1], activation='softmax'))  # Use softmax for multi-class classification

# Compile the model
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# Convert sparse matrices to dense NumPy arrays
X_train_dense = X_train_scaled.toarray()
X_test_dense = X_test_scaled.toarray()

# Train the model
model.fit(X_train_dense, y_train_encoded, epochs=10, batch_size=32, validation_split=0.2, verbose=1)

# Evaluate the model on the test set
predictions = model.predict(X_test_dense)
predicted_classes = np.argmax(predictions, axis=1)



Epoch 1/10
35493/35493 [==============================] - 74s 2ms/step - loss: 0.3764 - accuracy: 0.8814 - val_loss: 0.3739 - val_accuracy: 0.8816
Epoch 2/10
35493/35493 [==============================] - 83s 2ms/step - loss: 0.3737 - accuracy: 0.8817 - val_loss: 0.3734 - val_accuracy: 0.8816
Epoch 3/10
35493/35493 [==============================] - 83s 2ms/step - loss: 0.3735 - accuracy: 0.8817 - val_loss: 0.3739 - val_accuracy: 0.8816
Epoch 4/10
35493/35493 [==============================] - 82s 2ms/step - loss: 0.3733 - accuracy: 0.8817 - val_loss: 0.3734 - val_accuracy: 0.8816
Epoch 5/10
35493/35493 [==============================] - 81s 2ms/step - loss: 0.3733 - accuracy: 0.8817 - val_loss: 0.3733 - val_accuracy: 0.8816
Epoch 6/10
35493/35493 [==============================] - 85s 2ms/step - loss: 0.3732 - accuracy: 0.8817 - val_loss: 0.3730 - val_accuracy: 0.8816
Epoch 7/10
35493/35493 [==============================] - 78s 2ms/step - loss: 0.3732 - accuracy: 0.8817 - val_loss: 0

In [29]:
from sklearn.metrics import confusion_matrix

confussion_matrix=confusion_matrix(predicted_classes, predictions, labels=[0, 1, 2, 3, 4, 5,6, 7, 8, 9])

confussion_matrix



ValueError: Classification metrics can't handle a mix of binary and continuous-multioutput targets

In [ ]:
import numpy as np


def plot_confusion_matrix(cm,
                          target_names,
                          title='Confusion matrix',
                          cmap=None,
                          normalize=True):
    
    import matplotlib.pyplot as plt
    import numpy as np
    import itertools

    accuracy = np.trace(cm) / float(np.sum(cm))
    misclass = 1 - accuracy

    if cmap is None:
        cmap = plt.get_cmap('Blues')

    plt.figure(figsize=(8, 6))
    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title)
    plt.colorbar()

    if target_names is not None:
        tick_marks = np.arange(len(target_names))
        plt.xticks(tick_marks, target_names, rotation=45)
        plt.yticks(tick_marks, target_names)

    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]


    thresh = cm.max() / 1.5 if normalize else cm.max() / 2
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        if normalize:
            plt.text(j, i, "{:0.4f}".format(cm[i, j]),
                     horizontalalignment="center",
                     color="white" if cm[i, j] > thresh else "black")
        else:
            plt.text(j, i, "{:,}".format(cm[i, j]),
                     horizontalalignment="center",
                     color="white" if cm[i, j] > thresh else "black")


    plt.tight_layout()
    plt.ylabel('True label')
    plt.xlabel('Predicted label\naccuracy={:0.4f}; misclass={:0.4f}'.format(accuracy, misclass))
    plt.show()

plot_confusion_matrix(cm           = confussion_matrix, 
                      normalize    = False,
                      target_names = ['Analysis', 'Backdoor', 'DoS', 'Exploits', 'Fuzzers', 'Generic','Normal', 'Reconnaissance', 'Shellcode', 'Worms'],
                      title        = "Confusion Matrix")



In [ ]:
# from keras.models import Sequential
# from keras.layers import Dense, Dropout, BatchNormalization

# # Create a Sequential model
# model = Sequential()

# # Input layer
# model.add(Dense(128, activation='relu', input_dim=X_train_scaled.shape[1]))
# model.add(BatchNormalization())
# model.add(Dropout(0.5))  # Add dropout for regularization

# # Hidden layers
# model.add(Dense(256, activation='relu'))
# model.add(BatchNormalization())
# model.add(Dropout(0.5))

# model.add(Dense(128, activation='relu'))
# model.add(BatchNormalization())
# model.add(Dropout(0.5))

# # Output layer
# model.add(Dense(y_train_encoded.shape[1], activation='softmax'))

# # Compile the model
# model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# # Train the model
# model.fit(X_train_dense, y_train_encoded, epochs=20, batch_size=64, validation_split=0.2, verbose=1)

# # Evaluate the model on the test set
# predictions = model.predict(X_test_dense)
# predicted_classes = np.argmax(predictions, axis=1)


In [ ]:
import xgboost as xgb

In [ ]:

# Convert the training and testing data into DMatrix format
dtrain = xgb.DMatrix(X_train_scaled, label=y_train_encoded)
dtest = xgb.DMatrix(X_test_scaled, label=y_test_encoded)

param = {'max_depth': 10, 'eta': 1}
num_round = 10
bst = xgb.train(param, dtrain, num_round)

y_pred = bst.predict(dtest)
y_pred = np.round(y_pred).astype(int)

accuracy = accuracy_score(y_test_encoded, y_pred)
print("Accuracy: %.2f%%" % (accuracy * 100.0))

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Create a random forest classifier
rf_classifier = RandomForestClassifier(max_depth=10, n_estimators=10, random_state=42)

# Train the classifier
rf_classifier.fit(X_train_scaled, y_train_encoded)

# Make predictions on the test set
y_pred = rf_classifier.predict(X_test_scaled)

# Convert predicted values to integers
y_pred = y_pred.astype(int)

# Calculate accuracy
accuracy = accuracy_score(y_test_encoded, y_pred)
print("Accuracy: %.2f%%" % (accuracy * 100.0))


In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Create a decision tree classifier
dt_classifier = DecisionTreeClassifier(max_depth=10, random_state=42)

# Train the classifier
dt_classifier.fit(X_train_scaled, y_train_encoded)

# Make predictions on the test set
y_pred = dt_classifier.predict(X_test_scaled)

# Convert predicted values to integers
y_pred = y_pred.astype(int)

# Calculate accuracy
accuracy = accuracy_score(y_test_encoded, y_pred)
print("Accuracy: %.2f%%" % (accuracy * 100.0))


In [ ]:
from sklearn.preprocessing import LabelEncoder

# Use LabelEncoder to convert categorical labels to numerical labels
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)  # Assuming y_train is your original class labels


In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)


In [ ]:
model = Sequential()
model.add(Dense(64, activation='relu', input_dim=X_train_scaled.shape[1]))
model.add(Dense(32, activation='relu'))
model.add(Dense(y_train_encoded.shape[1], activation='softmax'))  # Use softmax for multi-class classification

# Compile the model
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# Convert sparse matrices to dense NumPy arrays
X_train_dense = X_train_scaled.toarray()
X_test_dense = X_test_scaled.toarray()

# Train the model
model.fit(X_train_dense, y_train_encoded, epochs=10, batch_size=32, validation_split=0.2, verbose=1)

# Evaluate the model on the test set
predictions = model.predict(X_test_dense)
predicted_classes = np.argmax(predictions, axis=1)



In [ ]:
from sklearn.metrics._classification import _check_targets

In [ ]:
plot_confusion_matrix(cm           = confussion_matrix, 
                      normalize    = False,
                      target_names = ['Analysis', 'Backdoor', 'DoS', 'Exploits', 'Fuzzers', 'Generic','Normal', 'Reconnaissance', 'Shellcode', 'Worms'],
                      title        = "Confusion Matrix")

In [ ]:
confussion_matrix=confusion_matrix(y_pred, predictions, labels=[0, 1, 2, 3, 4, 5,6, 7, 8, 9])

In [ ]:
y_pred = np.argmax(model.predict(X_test_dense), axis=-1)

cm = confusion_matrix(y_test_encoded, y_pred, labels=['Analysis', 'Backdoor', 'DoS', 'Exploits', 'Fuzzers', 'Generic','Normal', 'Reconnaissance', 'Shellcode', 'Worms'])
fig, ax = plt.subplots(figsize=(16, 12))
sns.heatmap(cm, annot=True, cmap='Blues', xticklabels=unique_labels, yticklabels=unique_labels)

plt.xlabel("Predicted Labels")
plt.ylabel("True Labels")
plt.title("Confusion Matrix")

In [ ]:
# # Convert one-hot encoded labels to binary
# y_test_binary = np.argmax(y_test_encoded, axis=1)

# # Evaluate the model
# accuracy = accuracy_score(y_test_binary, predicted_classes)
# conf_matrix = confusion_matrix(y_test_binary, predicted_classes)

# print("Accuracy:", accuracy)
# print("Confusion Matrix:")
# print(conf_matrix)


In [ ]:
from sklearn.model_selection import StratifiedKFold
from imblearn.over_sampling import RandomOverSampler
from sklearn import metrics
from keras.models import Sequential
from keras.layers import Convolution1D, MaxPooling1D, BatchNormalization, Bidirectional, LSTM, Reshape, Dropout, Dense, Activation


In [ ]:

oos_pred = []


In [ ]:
from sklearn.model_selection import StratifiedKFold
from imblearn.over_sampling import RandomOverSampler
oversample = SMOTE(sampling_strategy='minority')

In [ ]:
kfold = StratifiedKFold(n_splits=2,shuffle=True,random_state=42)
kfold.get_n_splits(X_train_dense,y_train_encoded)

In [ ]:

batch_size = 32
model = Sequential()
model.add(Convolution1D(64, kernel_size=64, padding="same", activation="relu", input_shape=(196, 1)))
model.add(MaxPooling1D(pool_size=10))
model.add(BatchNormalization())
model.add(Bidirectional(LSTM(64, return_sequences=False)))
model.add(Reshape((128, 1), input_shape=(128,)))
model.add(MaxPooling1D(pool_size=5))
model.add(BatchNormalization())
model.add(Bidirectional(LSTM(128, return_sequences=False)))
model.add(Dropout(0.6))
model.add(Dense(10))
model.add(Activation('softmax'))
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])


In [ ]:


for layer in model.layers:
    print(layer.output_shape)



In [ ]:
model.summary()

In [ ]:
import numpy as np


def plot_confusion_matrix(cm,
                          target_names,
                          title='Confusion matrix',
                          cmap=None,
                          normalize=True):
    
    import matplotlib.pyplot as plt
    import numpy as np
    import itertools

    accuracy = np.trace(cm) / float(np.sum(cm))
    misclass = 1 - accuracy

    if cmap is None:
        cmap = plt.get_cmap('Blues')

    plt.figure(figsize=(8, 6))
    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title)
    plt.colorbar()

    if target_names is not None:
        tick_marks = np.arange(len(target_names))
        plt.xticks(tick_marks, target_names, rotation=45)
        plt.yticks(tick_marks, target_names)

    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]


    thresh = cm.max() / 1.5 if normalize else cm.max() / 2
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        if normalize:
            plt.text(j, i, "{:0.4f}".format(cm[i, j]),
                     horizontalalignment="center",
                     color="white" if cm[i, j] > thresh else "black")
        else:
            plt.text(j, i, "{:,}".format(cm[i, j]),
                     horizontalalignment="center",
                     color="white" if cm[i, j] > thresh else "black")


    plt.tight_layout()
    plt.ylabel('True label')
    plt.xlabel('Predicted label\naccuracy={:0.4f}; misclass={:0.4f}'.format(accuracy, misclass))
    plt.show()

plot_confusion_matrix(cm           = confussion_matrix, 
                      normalize    = False,
                      target_names = ['Analysis', 'Backdoor', 'DoS', 'Exploits', 'Fuzzers', 'Generic','Normal', 'Reconnaissance', 'Shellcode', 'Worms'],
                      title        = "Confusion Matrix")



In [ ]:
# y_train_encoded = pd.DataFrame(y_train_encoded)  # Convert to DataFrame if not already
# y_train_encoded = y_train_encoded.fillna(y_train_encoded.mean())  # You can choose a different imputation method if needed


In [ ]:
#  for train_index, test_index in kfold.split(X_train_dense, label_column):

In [ ]:
y_train_encoded

In [ ]:
X_train_dense

In [ ]:
X_train_dense_df = pd.DataFrame(X_train_dense)

# Display the DataFrame
print(X_train_dense_df)

In [ ]:
for train_index, test_index in kfold.split(X_train_dense,y_train_encoded):
    train_X, test_X = combined_data_X.iloc[train_index], combined_data_X.iloc[test_index]
    train_y, test_y = y_train.iloc[train_index], y_train.iloc[test_index]
    
    print("train index:",train_index)
    print("test index:",test_index)
    print(train_y.value_counts())
    
    train_X_over,train_y_over= oversample.fit_resample(train_X, train_y)
    print(train_y_over.value_counts())
    
    x_columns_train = new_train_df.columns.drop('Class')
    x_train_array = train_X_over[x_columns_train].values
    x_train_1=np.reshape(x_train_array, (x_train_array.shape[0], x_train_array.shape[1], 1))
    
    dummies = pd.get_dummies(train_y_over) # Classification
    outcomes = dummies.columns
    num_classes = len(outcomes)
    y_train_1 = dummies.values
    
    x_columns_test = new_train_df.columns.drop('Class')
    x_test_array = test_X[x_columns_test].values
    x_test_2=np.reshape(x_test_array, (x_test_array.shape[0], x_test_array.shape[1], 1))
    
    dummies_test = pd.get_dummies(test_y) # Classification
    outcomes_test = dummies_test.columns
    num_classes = len(outcomes_test)
    y_test_2 = dummies_test.values
    
   
    model.fit(x_train_1, y_train_1,validation_data=(x_test_2,y_test_2), epochs=9)
    
    pred = model.predict(x_test_2)
    pred = np.argmax(pred,axis=1)
    y_eval = np.argmax(y_test_2,axis=1)
    score = metrics.accuracy_score(y_eval, pred)
    oos_pred.append(score)
    print("Validation score: {}".format(score))

In [ ]:
oos_pred
